# Loan Shark Escape Demo\nThis notebook compares a naive strategy against an LLM-guided strategy on `lse-medium`.

In [ ]:
import asyncio\nimport json\nfrom typing import Callable, Dict, Any\n\nimport pandas as pd\n\nfrom client import LoanSharkEscapeEnv\nfrom inference import _build_prompt, _call_llm, _parse_action\n\nAPI_BASE_URL = "http://localhost:8000"\nTASK_ID = "lse-medium"

In [ ]:
async def run_episode(policy: Callable[[Dict[str, Any]], int], label: str) -> Dict[str, Any]:\n    async with LoanSharkEscapeEnv(API_BASE_URL) as env:\n        reset_payload = await env.reset(TASK_ID)\n        obs = reset_payload.get("observation", reset_payload)\n\n        total_reward = 0.0\n        steps = 0\n        done = False\n\n        while not done and steps < 60:\n            action = int(policy(obs))\n            step_payload = await env.step({"action": action})\n            total_reward += float(step_payload.get("reward", 0.0))\n            done = bool(step_payload.get("done", False))\n            obs = step_payload.get("observation", step_payload)\n            steps += 1\n\n        grade = await env.evaluate()\n        state = await env.state()\n\n    return {\n        "agent": label,\n        "months": steps,\n        "total_reward": round(total_reward, 2),\n        "total_fees_paid": state.get("total_fees_paid", grade.get("total_fees_paid")),\n        "spiral_lock": state.get("spiral_lock", grade.get("spiral_lock_triggered")),\n        "all_loans_cleared": state.get("all_loans_cleared", grade.get("all_loans_cleared")),\n        "grader_reward": grade.get("reward"),\n        "grader": grade\n    }\n\ndef naive_policy(_obs: Dict[str, Any]) -> int:\n    return 1\n\ndef llm_policy(obs: Dict[str, Any]) -> int:\n    prompt = _build_prompt(obs)\n    text = _call_llm(prompt)\n    return _parse_action(text)

In [ ]:
naive_result = asyncio.run(run_episode(naive_policy, "Naive (always action=1)"))\nllm_result = asyncio.run(run_episode(llm_policy, "LLM-guided"))\n\ndf = pd.DataFrame([naive_result, llm_result])[\n    ["agent", "months", "total_fees_paid", "spiral_lock", "all_loans_cleared", "grader_reward"]\n]\n\nnaive_fees = float(naive_result.get("total_fees_paid") or 0)\nllm_fees = float(llm_result.get("total_fees_paid") or 0)\nfee_drop_pct = ((naive_fees - llm_fees) / naive_fees * 100.0) if naive_fees else 0.0\n\ndisplay(df)\nprint(f"Fee reduction: {fee_drop_pct:.2f}%")\nprint("\nNaive grader output:\n", json.dumps(naive_result["grader"], indent=2))\nprint("\nLLM grader output:\n", json.dumps(llm_result["grader"], indent=2))